# Example notebook to illustrate how to define a Training Plan and run an Experiment on Fedbiomed using Nipoppy Data

### Add imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.dataset import CustomDataset
from fedbiomed.common.datamanager import DataManager

from nipoppy import NipoppyDataRetriever

### Adding the dataset to node

Data is added to the node through the steps below:

1. `fedbiomed node --path ./CUSTOM-PATH-TO-NODE dataset add`

  * For each `Node`, select option `6` (Custom) to add your custom dataset to the node
  * Add a name and the tag for the dataset (tag is **`nipoppy`** in this example). Add the description.
  * Enter the path for your dataset location. In this example it should be `YOUR_WORK_DIR/nipoppy_example/my_dataset`. Path can be relative or absolute.
  
2. Data should be added. You can verify that your data has been added by executing `fedbiomed node --path YOUR-NODE-PATH dataset list`

3. Run the node using `fedbiomed node --path YOUR-NODE-PATH start`. Wait until you get `Starting task manager`. it means you are online.

Repeat the steps for all the nodes necessary. After the steps above are completed, you will be ready to train your model.

### Example Training Plan with Nipoppy Data using the Custom Dataset class

The Custom Dataset class in Fedbiomed allows the user to customize how the data is read and how a sample is retrieved. This is achieved by overriding the `read` and `get_item` functions in the class. An example can be seen in the cell below.

The `read` function uses `self.path` to read the data from the node. This path is the path given by the user in the previous step when adding the data to the node.

I wrote a placeholder linear model to show an example training using the data retrieved from Nipoppy Api.

In [ ]:
class NipoppyTrainingPlan(TorchTrainingPlan):

    # Model definition
    def init_model(self):
        model = self.MyCustomModel()
        return model

    # Here we define the custom dependencies that will be needed by our custom Dataloader
    def init_dependencies(self):
        deps = [
                "import pandas as pd",
                "import numpy as np",
                "from nipoppy import NipoppyDataRetriever",
                "from fedbiomed.common.dataset import CustomDataset",
            ]
        return deps

    # Torch modules class
    class MyCustomModel(nn.Module):

        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(3, 32)
            self.fc2 = nn.Linear(32, 16)
            self.fc3 = nn.Linear(16, 2)

        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            return F.log_softmax(self.fc3(x), dim=1)


    # Example to use the CustomDataset class to load Nipoppy data
    class NipoppyDataset(CustomDataset):
        """Custom Dataset for loading Nipoppy Data"""

        def read(self):
            """Read function in Fedbiomed is called once at the initialization of the Custom Dataset.
            
            It can be overridden similar to this example, to customize how the data is loaded and preprocessed.
            """

            # self.path variable is used to retrieve the dataset folder from the node
            self.api = NipoppyDataRetriever(self.path)

            self.df = self.api.get_tabular_data(
                phenotypes=[
                    "nb:Age",
                    "nb:Sex",
                    "nb:Diagnosis",
                    "snomed:859351000000102",  # MoCA
                ],
                derivatives=[
                    ("freesurfer", "7.3.2", "idp/fs_stats-0.2.1/fs7.3.2-aseg-volume.tsv"),
                    (
                        "freesurfer",
                        "7.3.2",
                        "idp/fs_stats-0.2.1/fs7.3.2-aparc-thickness.tsv",
                    ),
                ],
            )

            # Additional variables can be defined to be used in the training process
            # To illustrate an example, we convert the diagnosis column to categorical and create a dictionary
            # to map diagnosis strings to integer labels
            self.df['nb:Diagnosis'] = self.df['nb:Diagnosis'].astype('category')
            self.diagnosis_to_label = {
                cat: i for i, cat in enumerate(self.df['nb:Diagnosis'].cat.categories)
            }


        def get_item(self, index):
            """Get item function in Fedbiomed retrieves one sample from the dataset at each call/iteration.
            
            In the custom dataset, it can be overridden to customize how a sample will be retrieved.
            """

            # We select the features/columns from the sample, that we want to use in training the model
            data_cols = ['nb:Age', 'Left-Lateral-Ventricle', 'rh_insula_thickness']
            data = self.df.iloc[index][data_cols].to_numpy(dtype=float)
            data = torch.tensor(data, dtype=torch.float32)

            # Similarly we select the target column

            diagnosis_str = self.df.iloc[index]['nb:Diagnosis']
            target = torch.tensor(
                self.diagnosis_to_label[diagnosis_str],
                dtype=torch.long
            )

            return data, target
    

        def __len__(self):
            # If preferred, can be customized to only return the length of the selected columns,
            # instead of the length of the full dataframe
            return self.df.shape[0]


    # The training_data creates the Dataloader to be used for training in the
    # general class Torchnn of fedbiomed
    def training_data(self):
        dataset = self.NipoppyDataset()
        loader_arguments = { 'shuffle': True}
        return DataManager(dataset, **loader_arguments)

    # This function must return the loss to backward it
    def training_step(self, data, target):

        output = self.model().forward(data)
        loss   = torch.nn.functional.nll_loss(output, target)
        return loss

### Defining the experiment

The definition of an experiment entails defining all the necessary parameters for training. We can define batch_size or epochs, optimizer_args, model_args, the number of rounds we want to run Federated Learning and the aggregator we use for aggregating the results.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

training_args = {
    'loader_args': { 'batch_size': 32, }, 
    'optimizer_args': {
        'lr': 1e-3
    },
    'epochs': 1, 
    'dry_run': False,  
    'batch_maxnum': 100 # Fast pass for development : only use ( batch_maxnum * batch_size ) samples
}

tags =  ['nipoppy']
rounds = 5

exp = Experiment(tags=tags,
                 training_plan_class=NipoppyTrainingPlan,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 node_selection_strategy=None)

### Running the experiment

In [ ]:
exp.run()